In [ ]:
import os
import math
import numpy as np
import torch
import pandas as pd
from tqdm import tqdm
from datasets import Dataset
import evaluate
import nltk
import matplotlib.pyplot as plt
nltk.download("wordnet", quiet=True)

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    set_seed
)

# ======================
# CONFIG
# ======================
MODEL_NAME = "a"
DATA_FILE  = "/kaggle/input/versionfinal2/merged_datasetfinalv2 (2).csv"
OUTPUT_DIR = "/kaggle/working/t5-arabic-standard-finetune_simple"

MAX_INPUT_LENGTH  = 384
MAX_TARGET_LENGTH = 96
BATCH_SIZE = 4
EPOCHS = 3
LR = 3e-4  # Stable learning rate for fine-tuning
SEED = 42
WARMUP_RATIO = 0.1

set_seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ======================
# LOAD DATA
# ======================
def load_csv(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Error: The data file '{path}' was not found.")

    df = pd.read_csv(path)
    data = {"text": [], "summary": []}
    for _, r in tqdm(df.iterrows(), total=len(df), desc="Loading data"):
        t, s = str(r["text"]).strip(), str(r["summary"]).strip()
        if t and s:
            data["text"].append(t)
            data["summary"].append(s)
    return Dataset.from_dict(data)

dataset = load_csv(DATA_FILE)

# ======================
# TOKENIZER & PROMPT
# ======================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def prompt(x):
    return [f"النص: {t}\nالملخص:" for t in x]

# ======================
# PREPROCESS
# ======================
def preprocess(examples):
    inputs  = prompt(examples["text"])
    targets = examples["summary"]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding="max_length"
    )

    # Tokenize targets with text_target parameter
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets,
            max_length=MAX_TARGET_LENGTH,
            truncation=True,
            padding="max_length"
        )

    # Replace pad tokens with -100 to ignore in loss computation
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

dataset_tok = dataset.map(
    preprocess,
    batched=True,
    remove_columns=dataset.column_names
)

# Train/validation split
dataset_tok = dataset_tok.train_test_split(test_size=0.1, seed=SEED)
train_ds = dataset_tok["train"]
eval_ds  = dataset_tok["test"]

# ======================
# STANDARD MODEL (NO CUSTOM ARCHITECTURE) + CRITICAL FIXES
# ======================
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# 🔑 FIX #1: Disable cache at config level
model.config.use_cache = False

# 🔑 FIX #2: Override generate() to force use_cache=False in ALL generation calls
original_generate = model.generate

def safe_generate(*args, **kwargs):
    kwargs["use_cache"] = False  # Force-disable cache
    # Remove min_length if present (not always supported in all transformers versions)
    kwargs.pop("min_length", None)
    return original_generate(*args, **kwargs)

model.generate = safe_generate

# ======================
# DATA COLLATOR
# ======================
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=-100
)

# ======================
# METRICS
# ======================
bertscore_metric = evaluate.load("bertscore")
chrf_metric      = evaluate.load("chrf")
bleu_metric      = evaluate.load("bleu")
meteor_metric    = evaluate.load("meteor")

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    
    # Convert tensors to numpy if needed
    if isinstance(preds, torch.Tensor):
        preds = preds.cpu().numpy()
    if isinstance(labels, torch.Tensor):
        labels = labels.cpu().numpy()
    
    # Clean predictions: replace invalid tokens
    preds = np.where(preds < 0, tokenizer.pad_token_id, preds)
    preds = np.clip(preds.astype(np.int64), 0, tokenizer.vocab_size - 1)
    
    # Clean labels
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    labels = np.clip(labels.astype(np.int64), 0, tokenizer.vocab_size - 1)
    
    decoded_preds  = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # BERTScore
    bertscore = bertscore_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        lang="ar"
    )
    
    # chrF
    chrf = chrf_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )
    
    # BLEU
    bleu = bleu_metric.compute(
        predictions=decoded_preds,
        references=[[l] for l in decoded_labels]
    )
    
    # METEOR
    meteor = meteor_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )

    return {
        "bertscore_f1": np.mean(bertscore["f1"]),
        "chrf": chrf["score"],
        "bleu": bleu["bleu"],
        "meteor": meteor["meteor"],
    }

# ======================
# TRAINING ARGS (with safe generation config)
# ======================
# ======================
# TRAINING ARGS (CLEAN SINGLE-LINE PROGRESS)
# ======================
args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    eval_strategy="epoch",
    save_strategy="epoch",
    # 🔑 KEY SETTINGS FOR CLEAN PROGRESS:
    logging_strategy="epoch",      # Only log once per epoch (no step-by-step spam)
    disable_tqdm=False,            # Show compact single-line tqdm progress bar
    log_level="error",             # Suppress non-critical logs (warnings/info)
    log_level_replica="error",
    predict_with_generate=True,
    fp16=False,
    report_to="none",
    seed=SEED,
    logging_dir=f"{OUTPUT_DIR}/logs",
    save_safetensors=False
)

# ======================
# TRAINER (FIXED: 'args' not 'training_args')
# ======================
trainer = Seq2SeqTrainer(
    model=model,
    args=args,  # ✅ CORRECTED: was 'training_args' (undefined variable)
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)
    

print(f"🚀 Starting standard fine-tuning (no custom architecture)...")
print(f"   Model: {MODEL_NAME}")
print(f"   Training samples: {len(train_ds)}, Validation samples: {len(eval_ds)}")
trainer.train()

# ======================
# FINAL EVALUATION
# ======================
metrics = trainer.evaluate()
if "eval_loss" in metrics:
    metrics["perplexity"] = math.exp(metrics["eval_loss"])

print("\n" + "="*50)
print("FINAL EVALUATION METRICS")
print("="*50)
for k, v in sorted(metrics.items()):
    if isinstance(v, float):
        print(f"{k:30s}: {v:.4f}")
    else:
        print(f"{k:30s}: {v}")
print("="*50)

# ======================
# SAVE MODEL
# ======================
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\n✅ Model saved to: {OUTPUT_DIR}")

# ======================
# PLOTS
# ======================
log_history = trainer.state.log_history
train_loss, eval_loss = [], []
bertscore_f1, chrf_scores, bleu_scores = [], [], []

for log in log_history:
    if "loss" in log:
        train_loss.append(log["loss"])
    if "eval_loss" in log:
        eval_loss.append(log["eval_loss"])
        bertscore_f1.append(log.get("eval_bertscore_f1", 0))
        chrf_scores.append(log.get("eval_chrf", 0))
        bleu_scores.append(log.get("eval_bleu", 0))

epochs_range = range(1, len(eval_loss) + 1)
plots_dir = os.path.join(OUTPUT_DIR, "plots")
os.makedirs(plots_dir, exist_ok=True)

# Training Loss
plt.figure(figsize=(8, 4.5))
plt.plot(train_loss, linewidth=2)
plt.title("Training Loss", fontsize=14, fontweight='bold')
plt.xlabel("Training Step")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "train_loss.png"), dpi=150)
plt.close()

# Validation Loss
plt.figure(figsize=(8, 4.5))
plt.plot(epochs_range, eval_loss, marker='o', linewidth=2, markersize=6, color='#2E86AB')
plt.title("Validation Loss", fontsize=14, fontweight='bold')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "val_loss.png"), dpi=150)
plt.close()

# BERTScore F1
plt.figure(figsize=(8, 4.5))
plt.plot(epochs_range, bertscore_f1, marker='o', linewidth=2, markersize=6, color='#2E86AB')
plt.title("BERTScore F1", fontsize=14, fontweight='bold')
plt.xlabel("Epoch")
plt.ylabel("F1 Score")
plt.ylim(0, 1)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "bertscore_f1.png"), dpi=150)
plt.close()

print(f"\n✅ All plots saved to: {plots_dir}")